# Agentic Rag with Langgraph

In [9]:
uv add langchain.text_splitter

Note: you may need to restart the kernel to use updated packages.


d:\Agentic-AI-Practice\.venv\Scripts\python.exe: No module named uv


In [10]:
import os
from typing import TypedDict, List
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document



In [11]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [12]:
from openai import embeddings
import os
from langchain.chat_models import init_chat_model
llm=init_chat_model("groq:openai/gpt-oss-20b")

In [13]:
import os
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings

load_dotenv()

# Initialize OpenRouter Embedding Model
embeddings = OpenAIEmbeddings(
    model="nvidia/nemotron-3-embed-1b:free",
    openai_api_key=os.getenv("OPENROUTER_API_KEY"),
    openai_api_base="https://openrouter.ai/api/v1",
    check_embedding_ctx_length=False,  # Disables OpenAI-specific token counting
    tiktoken_enabled=False             # Disables tiktoken tokenizer for non-OpenAI models
)

# Test generating a single embedding
query_vector = embeddings.embed_query("Your text string goes here")
print(f"Embedding length: {len(query_vector)}")
print(f"Sample values: {query_vector[:5]}")


Embedding length: 2048
Sample values: [0.15173478424549103, -0.023006511852145195, -0.03310033306479454, -0.026551449671387672, -0.0043760063126683235]


# State Definition

In [14]:
class AgentState(TypedDict):
    question: str
    documents: List[Document]
    answer: str
    needs_retrieval: bool

In [17]:
!uv add langchain-chroma


Resolved 152 packages in 30ms
Checked 147 packages in 35ms


In [18]:
### Sample Document And VectorStore
from langchain_chroma import Chroma

# Sample documents for demonstration
sample_texts = [
    "LangGraph is a library for building stateful, multi-actor applications with LLMs. It extends LangChain with the ability to coordinate multiple chains across multiple steps of computation in a cyclic manner.",
    "RAG (Retrieval-Augmented Generation) is a technique that combines information retrieval with text generation. It retrieves relevant documents and uses them to provide context for generating more accurate responses.",
    "Vector databases store high-dimensional vectors and enable efficient similarity search. They are commonly used in RAG systems to find relevant documents based on semantic similarity.",
    "Agentic systems are AI systems that can take actions, make decisions, and interact with their environment autonomously. They often use planning and reasoning capabilities."
]

documents = [Document(page_content=text) for text in sample_texts]

## Create Chroma vector store (In-Memory)
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    collection_name="agentic_rag"
)

## Create retriever
# Note: In LangChain, pass 'k' via search_kwargs
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})


In [19]:
retriever

VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x0000017CFB4C1820>, search_kwargs={'k': 3})